## Data Download

Download wildlife-crossings from OSM (corresponding OSM tag: man_made = "wildlife_crossing") and export them to a GPKG for further use.

### Dependencies

In [8]:
import os

import geopandas as gpd
import requests as r
from shapely.geometry import LineString, MultiPolygon, Point, Polygon

from detect_wildlife_crossings.osm.queries import get_wildlife_crossing_query

### Config

In [ ]:
COUNTRIES_TO_PROCESS = ["CH"]  # List of country codes to process
GEODATA_OUTPUT_DIR = "../data/geodata/"  # Directory to save the output GPKGs, names will be generated as "<country>_wildlife_crossings_osm<bridges_string>.gpkg"
BRIDGES_ONLY = True  # whether to only consider bridges or tunnels and other as well (filters OSM on tag "bridge" = "yes")

In [ ]:
# these typically do not change
OVERPASS_API_URL = "https://overpass-api.de/api/interpreter"
CRS = "CRS:3035"  # target CRS for the output GPKGs
bridges_string = "_bridges_only" if BRIDGES_ONLY else ""
OVERPASS_TIMEOUT = 180  # timeout for overpass query in seconds

### Execution

In [19]:
for country in COUNTRIES_TO_PROCESS:
    print(f"Processing country: {country}")

    # Download wildlife crossings
    crossing_query = get_wildlife_crossing_query(
        country, timeout=OVERPASS_TIMEOUT, bridges_only=BRIDGES_ONLY
    )
    response = r.get(OVERPASS_API_URL, params={"data": crossing_query})
    response.raise_for_status()
    data = response.json()

    features = []
    for el in data["elements"]:
        tags = el.get("tags", {})

        # Node → Point
        if el["type"] == "node":
            geometry = Point(el["lon"], el["lat"])
            features.append({"id": el["id"], "geometry": geometry, **tags})

        # Way → LineString or Polygon
        elif el["type"] == "way" and "geometry" in el:
            coords = [(pt["lon"], pt["lat"]) for pt in el["geometry"]]
            if len(coords) >= 3 and coords[0] == coords[-1]:
                geometry = Polygon(coords)  # Closed → Polygon
            else:
                geometry = LineString(coords)  # Open → LineString
            features.append({"id": el["id"], "geometry": geometry, **tags})

        # Relation → MultiPolygon (if members have geometry)
        elif el["type"] == "relation" and "members" in el:
            polygons = []
            for member in el["members"]:
                if member["type"] == "way" and "geometry" in member:
                    coords = [(pt["lon"], pt["lat"]) for pt in member["geometry"]]
                    if len(coords) >= 3:
                        polygons.append(Polygon(coords))
            if polygons:
                geometry = MultiPolygon(polygons) if len(polygons) > 1 else polygons[0]
                features.append({"id": el["id"], "geometry": geometry, **tags})

    # --- Create GeoDataFrame ---
    if features:
        print(f"Found {len(features)} wildlife crossings.")
        gdf = gpd.GeoDataFrame(features, geometry="geometry", crs="EPSG:4326")
    else:
        print("No wildlife crossings found.")

    # --- Reproject to EPSG:3035 (ETRS89 / LAEA Europe) ---
    if not gdf.empty:
        gdf = gdf.to_crs(epsg=CRS.split(":")[1])
        print(f"Reprojected to CRS: {CRS}.")

    os.makedirs(GEODATA_OUTPUT_DIR, exist_ok=True)

    # Save the GeoDataFrame to a file
    output_file = os.path.join(
        GEODATA_OUTPUT_DIR,
        f"{country.lower()}_wildlife_crossings_osm{bridges_string}.gpkg",
    )
    gdf.to_file(output_file, driver="GPKG")
    print(f"Saved GeoDataFrame to {output_file}")

Processing country: CH


INFO:pyogrio._io:Created 18 records


Found 18 wildlife crossings.
Reprojected to CRS: EPSG:3035.
Saved GeoDataFrame to ../data/geodata/ch_wildlife_crossings_osm_bridges_only.gpkg
